# 05 — Gold dimensions

Build reporting dimensions and bridges only from available Silver entities. The notebook fails before writing if an expected source or column is absent. It does not fabricate source-system closure, reopen, or status-history data.


In [ ]:
AS_OF_DATE = ""  # Optional YYYY-MM-DD, passed by archive replay.
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
JOB_RUN_ID = ""  # Parent orchestration correlation ID.


In [ ]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 90_run_live_pipeline executes 00_setup_cfg before this child notebook.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
RUN_STARTED_AT = datetime.utcnow()


In [ ]:
def require_columns(source_table, required_columns):
    if not spark.catalog.tableExists(source_table):
        raise ValueError(f"Required Silver source is missing: {source_table}")
    actual = {field.name.lower() for field in spark.table(source_table).schema.fields}
    missing = sorted(set(required_columns) - actual)
    if missing:
        raise ValueError(f"{source_table} is missing required columns: {missing}")


def latest_dimension(source_table, target_table, key_columns, select_columns):
    """Write one current row per natural key from an available Silver source."""
    require_columns(source_table, key_columns + [source for source, _ in select_columns])
    frame = spark.table(source_table)
    order_columns = [
        F.col("export_date").cast("timestamp").desc_nulls_last(),
        F.col("_silver_load_ts").cast("timestamp").desc_nulls_last(),
    ]
    current = (
        frame.withColumn(
            "_gold_dimension_rank",
            F.row_number().over(Window.partitionBy(*key_columns).orderBy(*order_columns)),
        )
        .where(F.col("_gold_dimension_rank") == 1)
        .drop("_gold_dimension_rank")
    )
    dimension = current.select(*[
        F.col(source).alias(target) for source, target in select_columns
    ])
    (dimension.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    print(f"{target_table}: {dimension.count():,} rows from {source_table}")


def copy_bridge(source_table, target_table, required_columns, select_columns):
    require_columns(source_table, required_columns)
    bridge = spark.table(source_table).select(*[
        F.col(source).alias(target) for source, target in select_columns
    ]).dropDuplicates()
    (bridge.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(target_table))
    print(f"{target_table}: {bridge.count():,} rows from {source_table}")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SCHEMA}.dim_date AS
SELECT
  date_value AS date,
  YEAR(date_value) AS calendar_year,
  QUARTER(date_value) AS calendar_quarter,
  MONTH(date_value) AS calendar_month_number,
  DATE_FORMAT(date_value, 'MMMM') AS calendar_month_name,
  DATE_FORMAT(date_value, 'yyyy-MM') AS year_month,
  DAYOFWEEK(date_value) AS day_of_week_number,
  DATE_FORMAT(date_value, 'EEEE') AS day_of_week_name,
  DAYOFMONTH(date_value) AS day_of_month,
  CASE WHEN DAYOFWEEK(date_value) IN (1, 7) THEN false ELSE true END AS is_weekday,
  CURRENT_TIMESTAMP() AS gold_modelled_at
FROM (
  SELECT EXPLODE(SEQUENCE(DATE '2020-01-01', DATE '2035-12-31', INTERVAL 1 DAY)) AS date_value
)
""")

latest_dimension(
    "silver.holding_company", "gold.dim_holding_company", ["holding_company_id"],
    [("holding_company_id", "holding_company_id"), ("company_name", "company_name"),
     ("town_city", "town_city"), ("county", "county"), ("postcode", "postcode"),
     ("country", "country"), ("export_date", "source_export_date")],
)
latest_dimension(
    "silver.provider", "gold.dim_provider", ["provider_id"],
    [("provider_id", "provider_id"), ("holding_company_id", "holding_company_id"),
     ("provider_name", "provider_name"), ("provider_status", "provider_status"),
     ("town_city", "town_city"), ("county", "county"), ("postcode", "postcode"),
     ("country", "country"), ("qa_flag", "qa_flag"),
     ("export_date", "source_export_date")],
)
latest_dimension(
    "silver.provider_home", "gold.dim_provider_home", ["provider_home_id"],
    [("provider_home_id", "provider_home_id"), ("provider_id", "provider_id"),
     ("service_type", "service_type"), ("home_name", "home_name"),
     ("town_city", "town_city"), ("county", "county"), ("postcode", "postcode"),
     ("number_of_registered_beds", "registered_beds"), ("is_spot", "is_spot"),
     ("qa_flag", "qa_flag"), ("export_date", "source_export_date")],
)
latest_dimension(
    "silver.framework", "gold.dim_framework", ["framework_code"],
    [("framework_code", "framework_code"), ("framework_name", "framework_name"),
     ("placement_type", "placement_type"), ("start_date", "start_date"),
     ("end_date", "end_date"), ("export_date", "source_export_date")],
)
latest_dimension(
    "silver.framework_category", "gold.dim_framework_category", ["framework_category_id"],
    [("framework_category_id", "framework_category_id"), ("framework_code", "framework_code"),
     ("category_name", "category_name"), ("export_date", "source_export_date")],
)


In [ ]:
copy_bridge(
    "silver.provider_framework", "gold.bridge_provider_framework",
    ["provider_framework_id", "provider_id", "framework_code", "qa_flag", "export_date"],
    [("provider_framework_id", "provider_framework_id"), ("provider_id", "provider_id"),
     ("framework_code", "framework_code"), ("qa_flag", "qa_flag"),
     ("export_date", "source_export_date")],
)
copy_bridge(
    "silver.provider_sic_codes", "gold.bridge_provider_sic_code",
    ["provider_id", "sic_code", "export_date"],
    [("provider_id", "provider_id"), ("sic_code", "sic_code"),
     ("export_date", "source_export_date")],
)
latest_dimension(
    "silver.provider_submission_docs", "gold.dim_provider_submission_document", ["document_id"],
    [("document_id", "document_id"), ("submission_id", "submission_id"),
     ("s3_file_metadata_id", "s3_file_metadata_id"), ("document_name", "document_name"),
     ("document_type", "document_type"), ("expiry_date", "expiry_date"),
     ("last_updated", "last_updated"), ("next_review_date", "next_review_date"),
     ("service_type", "service_type"), ("home_id", "home_id"),
     ("start_date", "start_date"), ("export_date", "source_export_date")],
)

require_columns("silver.referral", ["placement_type", "referral_status"])
require_columns("silver.ipa", ["placement_type"])
placement_types = (
    spark.table("silver.referral").select(F.col("placement_type").alias("placement_type"))
    .unionByName(spark.table("silver.ipa").select(F.col("placement_type").alias("placement_type")))
    .where(F.col("placement_type").isNotNull() & (F.trim(F.col("placement_type")) != ""))
    .dropDuplicates()
    .withColumn("gold_modelled_at", F.current_timestamp())
)
(placement_types.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("gold.dim_placement_type"))

referral_statuses = (
    spark.table("silver.referral").select(F.col("referral_status").alias("referral_status"))
    .where(F.col("referral_status").isNotNull() & (F.trim(F.col("referral_status")) != ""))
    .dropDuplicates()
    .withColumn("gold_modelled_at", F.current_timestamp())
)
(referral_statuses.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("gold.dim_referral_status"))
print(f"Gold dimensions completed; AS_OF_DATE={AS_OF_DATE or 'latest'}; started={RUN_STARTED_AT.isoformat()}")
